# 04. Spatial distribution around genes

This notebook measures environmental-association signals and genomic features around the TSS and TTS of the top 5% MAGMA genes. It produces the panel data used for Figure 4A and Figures S9-S11.


# 04. 基因周围的空间分布

本 notebook 测量 MAGMA top 5% genes 的 TSS 和 TTS 周围的 environmental-association signals 与 genomic features，并生成 Figure 4A 以及 Figures S9-S11 所需的 panel data。


## Remote kernel initialization

The notebook runs against the remote project configuration. Verify the reported host, working directory, Python executable, environment, and expected kernelspec before continuing. This cell does not download data or start an analysis.


## 远程 kernel 初始化

本 notebook 根据远程项目配置运行。继续之前，请核对输出的 host、工作目录、Python executable、environment 和预期 kernelspec。本单元不会下载数据或启动分析。


In [ ]:
import os
import socket
import sys
from pathlib import Path

import yaml

%load_ext autoreload
%autoreload 2

launch_dir = Path.cwd().resolve()
project_file = launch_dir / "config" / "project.yaml"
if not project_file.is_file() and launch_dir.name == "notebooks":
    project_file = launch_dir.parent / "config" / "project.yaml"
if not project_file.is_file():
    raise FileNotFoundError("Copy config/project.example.yaml to config/project.yaml and complete it before running this notebook")
with project_file.open("r", encoding="utf-8") as stream:
    project_settings = yaml.safe_load(stream)
PROJECT_ROOT = Path(project_settings["remote"]["project_root"]).expanduser().resolve()
allowed_launch_dirs = {PROJECT_ROOT, PROJECT_ROOT / "notebooks"}
assert launch_dir in allowed_launch_dirs, f"Start Jupyter from the configured remote project root: {PROJECT_ROOT}"
assert PROJECT_ROOT.is_dir(), f"Remote project does not exist: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
expected_environment = project_settings["jupyter"]["kernel"]["environment"]["name"]
active_environment = os.environ.get("CONDA_DEFAULT_ENV", Path(sys.prefix).name)
if project_settings["jupyter"]["same_environment"]:
    server_environment = project_settings["jupyter"]["server"]["environment"]["name"]
    assert server_environment == expected_environment, (server_environment, expected_environment)
assert active_environment == expected_environment, (active_environment, expected_environment)
print({
    "host": socket.gethostname(),
    "working_directory": str(Path.cwd()),
    "python_executable": sys.executable,
    "environment": active_environment,
    "expected_kernelspec": project_settings["jupyter"]["kernel"]["kernelspec"],
})

## Run mode selection

Use `checkpoint` to read the published intermediate inputs or `full` to rebuild this notebook's intermediate products from source data. Outputs from the current run are always written to generated directories and never overwrite reference files.


## 运行模式选择

使用 `checkpoint` 读取已发布的中间输入；使用 `full` 从 source data 重建本 notebook 的中间产品。本次运行的输出始终写入 generated 目录，不覆盖 reference 文件。


In [ ]:
RUN_MODE = "checkpoint"
if RUN_MODE not in {"checkpoint", "full"}:
    raise ValueError("RUN_MODE must be checkpoint or full")
RUN_MODE

## Inputs and output contracts

All scientific inputs are resolved below `PROJECT_ROOT/data`. Published inputs are read from source or reference directories; newly built checkpoints and results are written only to generated directories. The following sections validate the notebook-specific schemas before interpreting results.


## 输入与输出契约

所有科学输入均从 `PROJECT_ROOT/data` 下解析。已发布输入从 source 或 reference 目录读取；新建 checkpoints 和 results 只写入 generated 目录。以下章节在解释结果前检查本 notebook 特有的 schemas。


## Scientific settings used in this notebook

Candidate-gene selection, spatial windows, summary quantile, null sampling, focal gradients, and the random seed are defined with the spatial analysis.


## 本 notebook 使用的科学设置

Candidate-gene selection、spatial windows、summary quantile、null sampling、重点环境梯度和随机种子在 spatial analysis 中直接定义。


In [ ]:
RANDOM_SEED = 5196
FOCAL_VARIABLES = ["gsl", "bio18", "al", "striga"]
CANDIDATE_GENE_FRACTION = 0.05
GROOVER_TOP_FRACTION = 0.05
FLANK_BP = 10000
WINDOW_BP = 500
STEP_BP = 10
SIGNAL_QUANTILE = 0.25
RANDOM_GENE_SETS = 1000


In [ ]:
import sys
import warnings
from functools import reduce
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from laenrich.config import load_project_paths
from laenrich.data import infer_groover_mutation_effects, source_path
from laenrich.io import read_table, validate_analysis_mode
from laenrich.plotting import plot_spatial_profile, save_figure
from laenrich.spatial import (
    empirical_profile_interval, high_effect_index, orient_variants_to_genes,
    sliding_signal_profile, sliding_track_profile,
)

paths = load_project_paths(PROJECT_ROOT / "config" / "project.yaml")
project_dir = paths.project_root
mode = validate_analysis_mode(RUN_MODE)
checkpoint_dir = paths.checkpoint_reference_dir if mode == "checkpoint" else paths.checkpoint_generated_dir
envgwas_dir = checkpoint_dir / "envgwas"
context_dir = checkpoint_dir / "contexts"
figure_data_dir = paths.result_generated_dir / "figure_data" / "figure4"
figure_dir = paths.result_generated_dir / "figures"
for directory in [figure_data_dir, figure_dir]:
    directory.mkdir(parents=True, exist_ok=True)
seed = int(RANDOM_SEED)
print({"mode": mode, "checkpoint_dir": str(checkpoint_dir)})


## 1. Candidate genes and strand-aware anchors

For each focal environmental variable, candidate genes are the top 5% of MAGMA P values. TSS and TTS anchors are derived from BTx623 v5.1 gene coordinates, and negative-strand genes are reversed so upstream and downstream directions are biologically aligned.


## 1. 候选基因与 strand-aware anchors

对每个重点环境变量，候选基因定义为 MAGMA P values 的 top 5%。TSS 和 TTS anchors 由 BTx623 v5.1 gene coordinates 获得；负链基因反向处理，使 upstream 与 downstream 的生物学方向一致。


In [ ]:
site_contexts = pd.read_hdf(context_dir / "site_contexts.h5", key="contexts").set_index("variant_id", drop=False)
gene_contexts = pd.read_hdf(context_dir / "gene_contexts.h5", key="contexts").set_index("gene_id", drop=False)
associations = read_table(envgwas_dir / "envgwas_associations.h5", ["variable_id", "variant_id", "chrom", "pos", "p_value"])
magma = read_table(envgwas_dir / "magma_gene_statistics.h5", ["variable_id", "gene_id", "p_value"])
focal_variables = FOCAL_VARIABLES
candidate_fraction = float(CANDIDATE_GENE_FRACTION)
candidate_records = []
for variable_id in focal_variables:
    table = magma.loc[magma["variable_id"].eq(variable_id)].sort_values("p_value", kind="stable")
    n_candidates = max(1, int(np.floor(len(table) * candidate_fraction)))
    candidate_records.append(table.head(n_candidates).assign(candidate_rank=np.arange(1, n_candidates + 1)))
candidate_genes = pd.concat(candidate_records, ignore_index=True)
candidate_genes.groupby("variable_id")["gene_id"].nunique()

## 2. Sliding-window envGWAS profiles

Signals are summarized across ±10 kb with a 500-bp window and 10-bp step. The first quartile of −log10(P) is calculated within each window and divided by the corresponding gene-level ±10-kb background value.


## 2. Sliding-window envGWAS profiles

在 ±10 kb 范围内使用 500-bp window 和 10-bp step 汇总信号。每个窗口计算 −log10(P) 的 first quartile，并除以对应基因 ±10-kb 范围的背景值。


In [ ]:
flank_bp = int(FLANK_BP)
window_bp = int(WINDOW_BP)
step_bp = int(STEP_BP)
signal_quantile = float(SIGNAL_QUANTILE)
rng = np.random.default_rng(seed)
profile_records = []
for variable_id in focal_variables:
    variant_table = associations.loc[associations["variable_id"].eq(variable_id)].copy()
    variant_table["signal"] = -np.log10(variant_table["p_value"])
    context_columns = [column for column in ["acr", "umr", "pi"] if column in site_contexts]
    variant_table = variant_table.merge(site_contexts[["variant_id", *context_columns]], on="variant_id", validate="one_to_one")
    observed_genes = candidate_genes.loc[candidate_genes["variable_id"].eq(variable_id), "gene_id"]
    gene_table = gene_contexts.loc[observed_genes, ["gene_id", "chrom", "start", "end", "strand"]]
    universe = gene_contexts.index.intersection(magma.loc[magma["variable_id"].eq(variable_id), "gene_id"])
    for anchor in ["tss", "tts"]:
        oriented = orient_variants_to_genes(variant_table, gene_table, anchor, flank_bp)
        observed_profile = sliding_signal_profile(oriented, "signal", flank_bp, window_bp, step_bp, signal_quantile)
        null_profiles = []
        for _ in range(int(RANDOM_GENE_SETS)):
            sampled_ids = rng.choice(universe.to_numpy(), size=len(gene_table), replace=False)
            sampled_genes = gene_contexts.loc[sampled_ids, ["gene_id", "chrom", "start", "end", "strand"]]
            sampled_oriented = orient_variants_to_genes(variant_table, sampled_genes, anchor, flank_bp)
            null_profiles.append(sliding_signal_profile(sampled_oriented, "signal", flank_bp, window_bp, step_bp, signal_quantile))
        interval = empirical_profile_interval(null_profiles)
        profile = observed_profile.merge(interval, on=["window_start", "window_end", "window_center"], validate="one_to_one")
        profile_records.append(profile.assign(variable_id=variable_id, anchor=anchor, track="envgwas_signal"))
        for context_column in context_columns:
            track = sliding_track_profile(oriented, context_column, flank_bp, window_bp, step_bp)
            track = track.rename(columns={"track_value": "relative_signal"})
            profile_records.append(track.assign(variable_id=variable_id, anchor=anchor, track=context_column))
spatial_profiles = pd.concat(profile_records, ignore_index=True, sort=False)
spatial_profiles.head()

## 3. Gene and UTR lengths, ACRs, UMRs, and nucleotide diversity

Gene/UTR extent distributions and the ACR, UMR, and nucleotide-diversity tracks use the same orientation and windows as the association profile, preventing coordinate or strand mismatches between panels.


## 3. Gene 与 UTR 长度、ACRs、UMRs 和 nucleotide diversity

Gene/UTR extent distributions 以及 ACR、UMR 和 nucleotide-diversity tracks 使用与关联 profile 相同的方向和窗口，从而避免 panels 之间的坐标或链方向不一致。


In [ ]:
gene_attribute_columns = [column for column in ["gene_length", "utr5_length", "utr3_length"] if column in gene_contexts]
gene_length_summary = candidate_genes.merge(
    gene_contexts[["gene_id", *gene_attribute_columns]].reset_index(drop=True), on="gene_id", validate="many_to_one"
).groupby("variable_id")[gene_attribute_columns].describe()
gene_length_summary

## 4. Groover promoter high-effect index

The three barcode-level Groover tables are read directly. Unnamed index columns are ignored and the bc/bc_x difference is handled explicitly. After barcode quality filtering, a sparse additive LSQR model separates single-mutation effects from multi-mutation constructs. Mutations in the top 5% of absolute effect within each assayed gene define the combined high-effect index.


## 4. Groover promoter high-effect index

直接读取三个 barcode-level Groover tables。分析忽略 unnamed index columns，并明确兼容 bc/bc_x 差异。完成 barcode quality filtering 后，使用 sparse additive LSQR model 从 multi-mutation constructs 中分离 single-mutation effects。每个实验基因内 absolute effect 位于 top 5% 的 mutations 用于构建 combined high-effect index。


In [ ]:
groover_inputs = {
    "PsbS": source_path(paths.source_dir, "groover_psbs"),
    "Raf1": source_path(paths.source_dir, "groover_raf1"),
    "SBPase": source_path(paths.source_dir, "groover_sbpase"),
}
effect_tables = []
audit_records = []
for gene_id, input_path in groover_inputs.items():
    effects, audit = infer_groover_mutation_effects(input_path, gene_id)
    effect_tables.append(effects)
    audit_records.append(audit)
promoter_mutations = pd.concat(effect_tables, ignore_index=True)
groover_audit = pd.DataFrame.from_records(audit_records)
promoter_index = high_effect_index(
    promoter_mutations, position_column="relative_position", effect_column="absolute_effect",
    gene_column="gene_id", top_fraction=GROOVER_TOP_FRACTION,
    flank_bp=FLANK_BP, window_bp=WINDOW_BP, step_bp=STEP_BP,
)
groover_audit


## 5. Figure 4A panel data and supplementary profiles

The GSL profile is the Figure 4A input. PHQ, AL, and STR profiles are saved as Figures S9-S11, and the complete long-format table is passed to notebook 05.


## 5. Figure 4A panel data 与补充 profiles

GSL profile 作为 Figure 4A 输入；PHQ、AL 和 STR profiles 保存为 Figures S9-S11；完整 long-format table 传递给 notebook 05。


In [ ]:
for variable_id in focal_variables:
    profile = spatial_profiles.loc[
        spatial_profiles["variable_id"].eq(variable_id) & spatial_profiles["anchor"].eq("tss") & spatial_profiles["track"].eq("envgwas_signal")
    ].copy()
    figure, axis = plot_spatial_profile(profile)
    axis.set_title(variable_id)
    figure.tight_layout()
    output_name = "figure4a_spatial_profile.pdf" if variable_id == "gsl" else f"figure_s_spatial_{variable_id}.pdf"
    save_figure(figure, figure_dir / output_name)
    plt.close(figure)
spatial_profiles.to_hdf(figure_data_dir / "spatial_profiles.h5", key="df", mode="w", format="table", index=False, complevel=9, complib="blosc:zstd")
candidate_genes.to_csv(figure_data_dir / "candidate_genes.tsv", sep="\t", index=False)
promoter_index.to_csv(figure_data_dir / "groover_high_effect_index.tsv", sep="\t", index=False)

## Manual recomputation commands

No external command is required by this notebook. It reads the completed context and envGWAS checkpoints and performs the strand-aware spatial summaries interactively.


## 手动重算命令

本 notebook 不需要外部命令。它读取已完成的 context 和 envGWAS checkpoints，并交互执行 strand-aware spatial summaries。


## Saved outputs and next step

Generated files remain in the remote working directory. Inspect the reported row counts, identifiers, quality-control summaries, and figure panels before proceeding to the next numbered notebook. External-tool commands described above remain manual steps.


## 已保存输出与下一步

生成的文件保留在远程工作目录中。进入下一本编号 notebook 前，请检查报告的行数、标识符、quality-control summaries 和 figure panels。以上 external-tool commands 仍需人工执行。
